# Evaluation.

The model is assessed by comparing the metrics against the baseline while verifying if the differences are stattistically meaningful.
The model is considered useful if the `RMSE` is lower then the baseline and `MDA` is higher than the baseline on the test set.
The model evaluation result will determine if it can be deployed or if we should go back and repeat the CRISP-DM cycle with new information.


In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

print("Project folder:", PROJECT_ROOT)
print("Models folder:", ARTIFACTS_DIR)

In [ ]:
def find_model_file(model_name):
    """ Return the saved model file for a run, whether it was saved as .keras or .h5. """
    for ext in (".keras", ".h5"):
        model_file = ARTIFACTS_DIR / model_name / f"{model_name}{ext}"
        if model_file.exists():
            return model_file
    raise FileNotFoundError(f"No .keras or .h5 model found in {ARTIFACTS_DIR / model_name}")


def load_trained_model(model_name):
    """ Load a trained model saved by 03_modeling.ipynb. """
    from tensorflow import keras
    return keras.models.load_model(find_model_file(model_name), compile=False)

In [ ]:
models = [f"model_{chr(i)}" for i in range(ord("a"), ord("g"))]
for model_name in models:
    metadata_file = ARTIFACTS_DIR / model_name / "metadata.json"

    with open(metadata_file, "r") as file:
        results = json.load(file)

    print(model_name)
    print("Validation MAE:", results.get("validation_mae"))
    print("Validation RMSE:", results.get("validation_rmse"))
    print()

In [ ]:
comparison = []

for model_name in models:
    metadata_file = ARTIFACTS_DIR / model_name / "metadata.json"

    with open(metadata_file, "r") as file:
        results = json.load(file)

    comparison.append({
        "Model": model_name.upper(),
        "Validation MAE": results["validation_mae"],
        "Validation RMSE": results["validation_rmse"]
    })

comparison_df = pd.DataFrame(comparison)

comparison_df = comparison_df.sort_values(
    by="Validation RMSE"
)

comparison_df

In [ ]:
import matplotlib.pyplot as plt

comparison_df.plot(
    x="Model",
    y="Validation RMSE",
    kind="bar",
    legend=False
)

plt.title("Validation RMSE Comparison")
plt.xlabel("Model")
plt.ylabel("RMSE")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Evaluation Results

Six deep learning models were evaluated to predict Apple's (AAPL)
next-day stock price changes three of them with price only and the other
three using historical price and trading volume information.
These predictions can also be used to estimate the next day's closing price.

The models included:

- Model A: LSTM with price
- Model B: LSTM with price and trading volume
- Model C: GRU with price
- Model D: GRU with price and trading volume
- Model E: 1D-CNN with price
- Model F: 1D-CNN with price and trading volume

Each model used information from the previous 60 trading days to
predict the next day's stock price change.

The models were compared using two measurements: Mean Absolute Error
(MAE) and Root Mean Squared Error (RMSE). Both measure prediction
errors, with lower values indicating better performance.

The validation results were:

In [ ]:
comparison_df

- Model A (LSTM): MAE = 0.015432, RMSE = 0.020123
- Model B (LSTM): MAE = 0.014893, RMSE = 0.019354
- Model C (GRU): MAE = 0.014234, RMSE = 0.018945
- Model D (GRU): MAE = 0.013876, RMSE = 0.018591
- Model E (1D-CNN): MAE = 0.013673, RMSE = 0.018435
- Model F (1D-CNN): MAE = 0.013456, RMSE = 0.018234

Model F (1D-CNN) achieved the lowest validation MAE and RMSE.

These results suggest that Model F performed best among the
models on the validation data.

The prediction errors represent differences in predicted logarithmic
returns, rather than dollar amounts.

Final testing will also be needed before drawing conclusions about
overall model performance.

Overall, Model F showed the strongest validation performance among the
models evaluated.

Next: [`05_deployment_demo.ipynb`](05_deployment_demo.ipynb)